# How to use this notebook:

# Input:
# - Annotated dataset (CSV exported from Label Studio)

# → Processing:
# 1. Formatting the annotated dataset (from Label Studio CSV to YOLO folder structure)
#    [no cleaning script, only file organization into the structure expected by YOLO]
# 2. Training the YOLO model

# Output:
# - A trained YOLO classification model


## Step 1: Preparing the dataset for YOLO model training


This script builds a YOLO-compatible folder structure (train, val, test) from a CSV file exported from manual annotation in Label Studio. Each image is sorted into a subfolder corresponding to its label.

A separate preprocessing step (not included here) involves a bash script to clean the annotation file by removing unnecessary digits from image filenames.

In an earlier experiment, the training pipeline was negatively affected by inconsistencies in file naming—specifically, the presence of whitespace characters in show titles. To ensure reproducibility, file names should remain clean and standardized.

In [11]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# set the paths for the CSV file, the annotated images from Label Studio, and the folder that will contain the structured dataset
csv_file_path = ''
images_dir = ''
output_dir = ''

# create output directories if needed
train_dir = os.path.join(output_dir, 'train')
val_dir = os.path.join(output_dir, 'val')
test_dir = os.path.join(output_dir, 'test')
for dir_path in [train_dir, val_dir, test_dir]:
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

# load annotations from the CSV file
annotations = pd.read_csv(csv_file_path)

# split the data into training, validation, and test sets
train_val, test = train_test_split(annotations, test_size=0.15, random_state=42)
train, val = train_test_split(train_val, test_size=0.1765, random_state=42)

# function to copy images into the appropriate directories
def copy_images(data, split):
    for _, row in data.iterrows():
        image_filename = row['image']
        choice = row['choice']
        
        # check that 'choice' is a valid value (not NaN, not empty)
        if pd.notna(choice) and choice != '':
            label = str(choice)  # convert to string
            
            # build the full path to the image
            image_path = os.path.join(images_dir, image_filename)
            
            # check that the file exists
            if not os.path.exists(image_path):
                print(f"Image file not found: {image_path}")
                continue
            
            # create the class directory if it doesn't exist
            class_dir = os.path.join(output_dir, split, label)
            if not os.path.exists(class_dir):
                os.makedirs(class_dir)
            
            # copy the image to the class directory
            dst_path = os.path.join(class_dir, image_filename)
            shutil.copy(image_path, dst_path)

# copy images for each split
copy_images(train, 'train')
copy_images(val, 'val')
copy_images(test, 'test')


## Step 2 : Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s-cls.pt")  # load a pretrained model (recommended for training)

dataset_custom = ""

# train
results = model.train(data=dataset_custom, epochs=100)


## Step 3 :Validate the model 

In [ ]:
# with defaults parameters
metrics = model.val()

## Step 4: Export the model 

In [ ]:
!pip install onnx


In [ ]:

model.export(format="onnx")